# 01. 데이터 전처리 및 공간 파생 변수 구축
도쿄 에어비앤비 숙소 데이터와 GIS 공간 데이터를 결합하여 핵심 파생 변수를 생성합니다.

In [ ]:
import pandas as pd
import geopandas as gpd
from google.colab import files

## 1. 데이터 불러오기 및 기본 필터링 (2~4인용)
숙소 인원수 차이가 너무 많아 2~4인용 숙소로 제한함

In [ ]:
df = pd.read_csv('/listings.csv')
df = df[(df['accommodates'] >= 2) & (df['accommodates'] <= 4)].copy()
df = df.dropna(subset=['latitude', 'longitude']).copy()

# 지도 데이터로 변환 (EPSG:3857 - 미터 단위 거리 계산용)
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.longitude, df.latitude),
    crs="EPSG:4326"
).to_crs(epsg=3857)

## 2. 역 데이터 결합 (거리 및 500m 내 개수)

In [ ]:
stations = gpd.read_file('/N02-25_Station.geojson')
stations = stations.to_crs(epsg=3857)
stations['geometry'] = stations.centroid
stations_clean = stations[['geometry']].copy()

# 가장 가까운 역까지 거리
gdf = gpd.sjoin_nearest(gdf, stations_clean, how='left', distance_col='dist_to_station')
gdf = gdf[~gdf.index.duplicated(keep='first')]
gdf = gdf.drop(columns=['index_right'], errors='ignore')

# 500m 내 역 개수
gdf_buffer = gdf.copy()
gdf_buffer = gdf_buffer.set_geometry(gdf.geometry.buffer(500))
joined_stations = gpd.sjoin(gdf_buffer, stations_clean, how='inner', predicate='contains')
gdf['stations_within_500m'] = joined_stations.groupby(joined_stations.index).size()
gdf['stations_within_500m'] = gdf['stations_within_500m'].fillna(0)

## 3. 주요 중심지까지의 거리 계산

In [ ]:
poi_df = pd.DataFrame({
    'name': ['Shinjuku', 'Tokyo', 'Shibuya'],
    'lon': [139.70003485471355, 139.7662611338308, 139.7012281007357],
    'lat': [35.689563103340696, 35.68136929058714, 35.658931748846044]
})

poi_gdf = gpd.GeoDataFrame(
    poi_df,
    geometry=gpd.points_from_xy(poi_df.lon, poi_df.lat),
    crs="EPSG:4326"
).to_crs(epsg=3857)

# 거리 계산
for _, row in poi_gdf.iterrows():
    gdf[f"dist_to_{row['name']}"] = gdf.geometry.distance(row.geometry)

## 4. 지가(토지 가격) 데이터 결합

In [ ]:
land = gpd.read_file('/L01-26_13.geojson')
land = land.to_crs(epsg=3857)

land_clean = land[['L01_008', 'geometry']].copy()
land_clean = land_clean.rename(columns={'L01_008': 'land_price'})

gdf = gpd.sjoin_nearest(gdf, land_clean, how='left', distance_col='dist_to_land')
gdf = gdf[~gdf.index.duplicated(keep='first')]
gdf = gdf.drop(columns=['index_right'], errors='ignore')

## 5. 최종 변수 추출 및 저장

In [ ]:
final_cols = [
    'id', 'name', 'listing_url', 'price', 'room_type', 'property_type',
    'accommodates', 'bedrooms', 'beds', 'bathrooms_text', 'amenities',
    'minimum_nights', 'neighbourhood_cleansed', 'latitude', 'longitude',
    'review_scores_rating', 'review_scores_value', 'review_scores_cleanliness',
    'review_scores_location', 'review_scores_checkin', 'review_scores_communication',
    'review_scores_accuracy', 'number_of_reviews', 'host_is_superhost',
    'host_response_rate', 'host_acceptance_rate', 'host_identity_verified',
    'calculated_host_listings_count', 'availability_365', 'instant_bookable',
    'dist_to_station', 'stations_within_500m',
    'dist_to_Shinjuku', 'dist_to_Tokyo', 'dist_to_Shibuya',
    'land_price', 'dist_to_land'
]
 # 공간 파생 변수
    'dist_to_station',
    'stations_within_500m',
    'dist_to_Shinjuku',
    'dist_to_Tokyo',
    'dist_to_Shibuya',
    'land_price',
    'dist_to_land'

existing_cols = [col for col in final_cols if col in gdf.columns]
final_df = pd.DataFrame(gdf[existing_cols])

# 결과 출력 및 다운로드
print("중심지 거리 샘플:")
print(final_df[['dist_to_Shinjuku', 'dist_to_Tokyo', 'dist_to_Shibuya']].head())


file_name = 'final_airbnb_with_spatial_and_land_features.csv'
final_df.to_csv(file_name, index=False)
files.download(file_name)